# Global PIQA Evaluation (isQnA-aware) — LaMini-GPT-1.5B
## Sinhala Script Robustness — Unicode vs Romanized

**Model:** `MBZUAI/LaMini-GPT-1.5B`  
**Prompt template:** `T1_direct` (isQnA-aware)  
**Reasoning mode:** `none`  
**8-bit quantization:** `False`

### isQnA-aware prompting

The `global_piqa.jsonl` dataset contains an `isQnA` boolean flag per item:

| `isQnA` | Prompt style | Domain hint |
|---------|--------------|-------------|
| `true`  | MCQ — `"This is a multiple-choice question related to the Sri Lankan Culture. Choose the correct or most appropriate answer from answers {range}. ... Respond with only the number ({range}) and nothing else, exactly like the examples below."` | `Sri Lankan Culture` |
| `false` | Paragraph-completion — `"Out of the given {n} answers, which one is the best option to complete the given text? ... Respond with only the number ({range}) and nothing else, exactly like the examples below."` | — |


## 1. Configuration

In [1]:
# ============================================================
# MODEL CONFIGURATION
# ============================================================

MODEL_ID    = "MBZUAI/LaMini-GPT-1.5B"
MODEL_LABEL = "LaMini-GPT-1.5B"
PROMPT_TEMPLATE = "T1_direct"  # isQnA-aware T1_direct

# Reasoning-mode flag
# Options: "none" | "no_think" | "enable_thinking_false"
REASONING_MODE = "none"

MAX_NEW_TOKENS   = 40
TEMPERATURE      = 0.0
MAX_INPUT_TOKENS = 4096  # upper-bound safety cap; real value clamped to model's actual context limit after loading (see Cell 4)

LOAD_IN_8BIT = False
RESUME       = True
TIME_LIMIT_HOURS = 11.5

# Dataset path (update if Kaggle dataset mount differs)
DATA_DIR = "/kaggle/input/datasets/anon-owner-a/piqa-new"

import re
OUTPUT_PREFIX = re.sub(r"[^a-z0-9]+", "_", MODEL_LABEL.lower()).strip("_")

print(f"Model:          {MODEL_ID}")
print(f"Label:          {MODEL_LABEL}")
print(f"Template:       {PROMPT_TEMPLATE} (isQnA-aware)")
print(f"Reasoning mode: {REASONING_MODE}")
print(f"8-bit:          {LOAD_IN_8BIT}")
print(f"Resume:         {RESUME}")
print(f"Time limit:     {TIME_LIMIT_HOURS} hours")
print(f"Output:         {OUTPUT_PREFIX}_global_piqa.csv")

Model:          MBZUAI/LaMini-GPT-1.5B
Label:          LaMini-GPT-1.5B
Template:       T1_direct (isQnA-aware)
Reasoning mode: none
8-bit:          False
Resume:         True
Time limit:     11.5 hours
Output:         lamini_gpt_1_5b_global_piqa.csv


## 2. Environment Setup & Hardware Verification

In [2]:
import sys
import os

!{sys.executable} -m pip install --upgrade --quiet transformers accelerate huggingface_hub bitsandbytes

import torch
import json
import csv
import time
import re
import transformers
from pathlib import Path
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

device_count = torch.cuda.device_count()
print(f"PyTorch {torch.__version__} | transformers {transformers.__version__} | {device_count} GPU(s)")
for i in range(device_count):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
    print(f"  cuda:{i} -> {name} ({mem:.1f} GB)")
assert device_count > 0, "No GPU detected. Enable GPU T4 x2 in Session options."

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 100.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 45.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 113.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 109.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
PyTorch 2.10.0+cu128 | transformers 5.17.0 | 2 GPU(s)
  cuda:0 -> Tesla 

## 3. HuggingFace Authentication

In [3]:
from huggingface_hub import login

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
    login(token=HF_TOKEN)
    print("Authenticated via Kaggle Secrets.")
except Exception as e:
    HF_TOKEN = ""   # Paste your token here if Kaggle Secrets is unavailable
    if HF_TOKEN:
        login(token=HF_TOKEN)
        print("Authenticated via explicit token.")
    else:
        print(f"No HF_TOKEN found ({e}). Public models will still work.")

No HF_TOKEN found (Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 134858644 and label HF_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.). Public models will still work.


## 4. Load Model & Tokenizer

In [4]:
print(f"Loading {MODEL_ID}...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

has_template = hasattr(tokenizer, "chat_template") and tokenizer.chat_template is not None
print(f"Chat template available: {has_template}")
if not has_template:
    print("WARNING: this model has no chat template. It will be prompted with raw text.")
if not has_template and REASONING_MODE != "none":
    print(f"WARNING: REASONING_MODE={REASONING_MODE!r} but no chat template -- setting has no effect.")

model_kwargs = dict(
    device_map="auto",
    token=HF_TOKEN if HF_TOKEN else None,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
if LOAD_IN_8BIT:
    from transformers import BitsAndBytesConfig
    model_kwargs["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
else:
    model_kwargs["torch_dtype"] = torch.float16

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
model.eval()
model.config.use_cache = True


# --- Dynamic input-token cap (Change 1) ---
# Read the model's actual context limit and clamp MAX_INPUT_TOKENS to it.
# LaMini-GPT-1.5B is a fine-tuned gpt2-xl with n_positions=1024; passing
# position_ids >= 1024 to the GPU causes a CUDA device-side assert.
_model_ctx = (
    getattr(model.config, "max_position_embeddings", None)
    or getattr(model.config, "n_positions", None)
    or getattr(tokenizer, "model_max_length", None)
)
if not _model_ctx or _model_ctx > 100_000:  # tokenizer placeholder is often 1e30
    _model_ctx = 1024

_SAFETY_MARGIN = 8
MAX_INPUT_TOKENS = max(1, min(MAX_INPUT_TOKENS, _model_ctx - MAX_NEW_TOKENS - _SAFETY_MARGIN))
tokenizer.truncation_side = "left"  # preserve the trailing Answer: cue under truncation
print("Model loaded successfully.")
print(f"Precision: {model.dtype}")
print(f"Model context limit: {_model_ctx} tokens")
print(f"Effective MAX_INPUT_TOKENS (after clamp): {MAX_INPUT_TOKENS}")
if hasattr(model, "hf_device_map"):
    print(f"Device map: {model.hf_device_map}")

Loading MBZUAI/LaMini-GPT-1.5B...


config.json:   0%|          | 0.00/984 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/788 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/581 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: MBZUAI/LaMini-GPT-1.5B
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...47}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded successfully.
Precision: torch.float16
Model context limit: 1024 tokens
Effective MAX_INPUT_TOKENS (after clamp): 976
Device map: {'transformer.wte': 0, 'lm_head': 0, 'transformer.wpe': 0, 'transformer.drop': 0, 'transformer.h.0': 0, 'transformer.h.1': 0, 'transformer.h.2': 0, 'transformer.h.3': 0, 'transformer.h.4': 0, 'transformer.h.5': 0, 'transformer.h.6': 0, 'transformer.h.7': 0, 'transformer.h.8': 0, 'transformer.h.9': 0, 'transformer.h.10': 0, 'transformer.h.11': 0, 'transformer.h.12': 0, 'transformer.h.13': 0, 'transformer.h.14': 0, 'transformer.h.15': 0, 'transformer.h.16': 0, 'transformer.h.17': 0, 'transformer.h.18': 0, 'transformer.h.19': 0, 'transformer.h.20': 0, 'transformer.h.21': 0, 'transformer.h.22': 1, 'transformer.h.23': 1, 'transformer.h.24': 1, 'transformer.h.25': 1, 'transformer.h.26': 1, 'transformer.h.27': 1, 'transformer.h.28': 1, 'transformer.h.29': 1, 'transformer.h.30': 1, 'transformer.h.31': 1, 'transformer.h.32': 1, 'transformer.h.33': 1, 'tr

## 5. Load Global PIQA Dataset

In [5]:
def load_jsonl(filepath):
    """Load a JSONL file into a list of dicts."""
    data = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

piqa_data = load_jsonl(os.path.join(DATA_DIR, "global_piqa.jsonl"))

print(f"Global PIQA: {len(piqa_data):,} items")

# isQnA distribution
n_qna  = sum(1 for it in piqa_data if it.get("isQnA", False))
n_comp = len(piqa_data) - n_qna
print(f"  isQnA=True  (MCQ format):                  {n_qna:3d} items")
print(f"  isQnA=False (paragraph-completion format): {n_comp:3d} items")

Global PIQA: 100 items
  isQnA=True  (MCQ format):                   40 items
  isQnA=False (paragraph-completion format):  60 items


## 6. isQnA-Aware Prompt Templates & Label Extraction

- **`isQnA=True`** → MCQ format with domain `Sri Lankan Culture`  
- **`isQnA=False`** → Paragraph-completion format (no domain)  


In [6]:
# ============================================================
# isQnA-AWARE PROMPT BUILDER  (T1_direct, zero-shot)
# ============================================================
#
#  isQnA=True  -> MCQ format with domain='Sri Lankan Culture'
#  isQnA=False -> paragraph-completion format (no domain)

NO_REPEAT_INSTRUCTION = "Do not repeat the question or text in your response."
PIQA_DOMAIN = "Sri Lankan Culture"


def _humanize_domain(domain):
    return str(domain).replace("_", " ")


def _mcq_options_block(options):
    digit_labels = [str(i + 1) for i in range(len(options))]
    options_str = "\n".join(f"{d}. {opt}" for d, opt in zip(digit_labels, options))
    return options_str, digit_labels


def build_piqa_prompt(question, options, is_qna):
    """Build a T1_direct prompt for a Global PIQA item.

    Parameters
    ----------
    question : str  -- the Sinhala question / paragraph stem
    options  : list -- answer option strings
    is_qna   : bool -- True -> MCQ w/ domain; False -> paragraph-completion
    """
    options_str, digit_labels = _mcq_options_block(options)
    n = len(options)
    valid_range = "/".join(digit_labels)

    if is_qna:
        # MCQ format -- domain hint: Sri Lankan Culture (Change 3: tightened T1_direct)
        prompt = (
            f"Multiple-choice question ({_humanize_domain(PIQA_DOMAIN)}). "
            f"Reply with ONLY the number ({valid_range}) -- no words, no explanation.\n"
            f"Question: {question}\n"
            f"{options_str}\n"
            f"Answer (number {valid_range} only):"
        )
    else:
        # Paragraph-completion format -- with Sri Lankan Culture domain (Change 3: tightened T1_direct)
        prompt = (
            f"Text completion ({_humanize_domain(PIQA_DOMAIN)}). "
            f"Reply with ONLY the number ({valid_range}) -- no words, no explanation.\n"
            f"{question}\n"
            f"{options_str}\n"
            f"Answer (number {valid_range} only):"
        )

    return prompt, digit_labels


# ============================================================
# LABEL EXTRACTION
# ============================================================

def _extract_token(generated_text, valid_tokens, keyword_pattern):
    """Boundary-safe extractor for digit tokens."""
    if not generated_text:
        return "INVALID"
    text = generated_text.strip()
    pattern_tokens = "|".join(re.escape(t) for t in valid_tokens)
    is_digit = valid_tokens and valid_tokens[0].isdigit()
    boundary     = r"(?<!\d)" if is_digit else r"\b"
    end_boundary = r"(?!\d)"  if is_digit else r"\b"

    prefix_match = re.match(
        rf"^(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)?"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if prefix_match:
        return prefix_match.group(1).upper()

    body_match = re.search(
        rf"(?:(?:the\s+)?(?:{keyword_pattern})\s*(?:is|=|:)?\s*)"
        rf"[\*\(#\[\s]*{boundary}({pattern_tokens}){end_boundary}",
        text, re.IGNORECASE
    )
    if body_match:
        return body_match.group(1).upper()

    standalone = re.search(rf"{boundary}({pattern_tokens}){end_boundary}", text, re.IGNORECASE)
    if standalone:
        return standalone.group(1).upper()

    return "INVALID"


def extract_mcq_choice(generated_text, digit_labels):
    return _extract_token(generated_text, digit_labels, r"correct\s+answer|answer|option|choice")


def digit_to_letter(digit):
    """1->A, 2->B, ... INVALID passes through."""
    if digit.isdigit() and 1 <= int(digit) <= 26:
        return chr(64 + int(digit))
    return digit


print("isQnA-aware prompt builder and label extractors defined.")

# Sanity-check both branches
p_qna, d = build_piqa_prompt("Sample question?", ["opt1", "opt2"], is_qna=True)
print(f"\n--- Sample isQnA=True Prompt ---\n{p_qna}")
p_comp, d = build_piqa_prompt("Sample text stem,", ["opt1", "opt2"], is_qna=False)
print(f"\n--- Sample isQnA=False Prompt ---\n{p_comp}")

isQnA-aware prompt builder and label extractors defined.

--- Sample isQnA=True Prompt ---
Multiple-choice question (Sri Lankan Culture). Reply with ONLY the number (1/2) -- no words, no explanation.
Question: Sample question?
1. opt1
2. opt2
Answer (number 1/2 only):

--- Sample isQnA=False Prompt ---
Text completion (Sri Lankan Culture). Reply with ONLY the number (1/2) -- no words, no explanation.
Sample text stem,
1. opt1
2. opt2
Answer (number 1/2 only):


## 7. Generation Engine
`REASONING_MODE = "none"` — applied identically to both script conditions.


In [7]:
@torch.inference_mode()
def generate_response(prompt, max_new_tokens=MAX_NEW_TOKENS, temperature=TEMPERATURE):
    """Generate a response from the instruct model."""
    has_template = hasattr(tokenizer, "apply_chat_template") and getattr(tokenizer, "chat_template", None)
    try:
        if has_template:
            if REASONING_MODE == "no_think":
                messages = [
                    {"role": "system", "content": "/no_think"},
                    {"role": "user",   "content": prompt},
                ]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            elif REASONING_MODE == "enable_thinking_false":
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
                )
            else:  # 'none' -- standard
                messages = [{"role": "user", "content": prompt}]
                formatted_prompt = tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
        else:
            formatted_prompt = prompt

        device = next(model.parameters()).device
        inputs = tokenizer(
            formatted_prompt, return_tensors="pt", truncation=True, max_length=MAX_INPUT_TOKENS
        ).to(device)
        input_len = inputs["input_ids"].shape[1]

        gen_kwargs = {
            "max_new_tokens": max_new_tokens,
            "do_sample": temperature > 0,
            "pad_token_id": tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id,
        }
        if temperature > 0:
            gen_kwargs["temperature"] = temperature

        outputs = model.generate(**inputs, **gen_kwargs)
        generated_ids = outputs[0][input_len:]
        generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return generated_text, input_len, len(generated_ids)
    except Exception as e:
        torch.cuda.empty_cache()
        print(f"\n[WARNING] Generation error: {e}")
        return f"ERROR: {e}", 0, 0


print(f"Generation engine initialized. REASONING_MODE = {REASONING_MODE!r}")

# Quick sanity-check
_sp, _ = build_piqa_prompt("Sample question?", ["opt1", "opt2"], is_qna=True)
_so, _, _ = generate_response(_sp)
print(f"Sample output (isQnA=True):  {repr(_so[:100])}")
_sp2, _ = build_piqa_prompt("Sample text stem,", ["opt1", "opt2"], is_qna=False)
_so2, _, _ = generate_response(_sp2)
print(f"Sample output (isQnA=False): {repr(_so2[:100])}")
if REASONING_MODE != "none" and "<think>" in (_so + _so2).lower():
    print("WARNING: <think> block still present -- suppression flag did not take effect.")

Generation engine initialized. REASONING_MODE = 'none'


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Sample output (isQnA=True):  ' opt1'
Sample output (isQnA=False): '\n1. opt1\n2. opt2'


## 8. Pilot Run

In [8]:
print("=" * 75)
print("PILOT RUN -- 5 items  (mix of isQnA=True and isQnA=False)")
print(f"Prompt template: T1_direct (isQnA-aware)")
print(f"Reasoning mode:  {REASONING_MODE}")
print("=" * 75)

for item in piqa_data[:5]:
    for script in ["unicode", "romanized"]:
        s = item[script]
        is_qna = item.get("isQnA", False)
        prompt, digits = build_piqa_prompt(s["text"], s["options"], is_qna=is_qna)
        gen_text, n_in, n_out = generate_response(prompt)
        pred_digit = extract_mcq_choice(gen_text, digits)
        pred_label = digit_to_letter(pred_digit)
        correct = "\u2713" if pred_label == item["label"] else "\u2717"
        fmt = "MCQ " if is_qna else "COMP"
        print(f"  [{script:9}] [{fmt}] id={item['id']} | gold={item['label']} pred={pred_digit}->{pred_label} {correct} | raw={gen_text[:60]!r}")

print("\n" + "=" * 75)
print("Pilot complete. Check output above before running Section 10.")
print("=" * 75)

PILOT RUN -- 5 items  (mix of isQnA=True and isQnA=False)
Prompt template: T1_direct (isQnA-aware)
Reasoning mode:  none
  [unicode  ] [MCQ ] id=piqa_0001 | gold=A pred=INVALID->INVALID ✗ | raw=' හේන කපා, වවා, රැ�'
  [romanized] [MCQ ] id=piqa_0001 | gold=A pred=1->A ✓ | raw=' 1'
  [unicode  ] [COMP] id=piqa_0002 | gold=B pred=INVALID->INVALID ✗ | raw=' ගරුඬ යකා ය\nගරුඬ'
  [romanized] [COMP] id=piqa_0002 | gold=B pred=INVALID->INVALID ✗ | raw=' Garunda yakaa kiyaa ya.'
  [unicode  ] [MCQ ] id=piqa_0003 | gold=A pred=INVALID->INVALID ✗ | raw=' රුදුරු පෙතංගො�'
  [romanized] [MCQ ] id=piqa_0003 | gold=A pred=1->A ✓ | raw=' 1. ruduru pethangoda uyanata giya pasuwa napuru una katuwa '
  [unicode  ] [COMP] id=piqa_0004 | gold=A pred=INVALID->INVALID ✗ | raw=' සුදු වෙන්නට සි�'
  [romanized] [COMP] id=piqa_0004 | gold=A pred=1->A ✓ | raw=' 1. Suuryagrahanayak weelaawee ya.\nAnswer (number 2/2 only):'
  [unicode  ] [COMP] id=piqa_0005 | gold=B pred=INVALID->INVALID ✗ | raw=' භ්න්නාට ලණුවක �'
  

## 9. Evaluation Runner

In [9]:
MCQ_FIELDNAMES = [
    "id", "gold_label", "is_qna",
    "unicode_pred_digit", "unicode_pred_label", "unicode_correct", "unicode_raw_output",
    "unicode_n_input_tokens", "unicode_n_output_tokens",
    "romanized_pred_digit", "romanized_pred_label", "romanized_correct", "romanized_raw_output",
    "romanized_n_input_tokens", "romanized_n_output_tokens",
]

GLOBAL_START_TIME = time.time()

def _check_time_limit():
    return (time.time() - GLOBAL_START_TIME) / 3600.0 >= TIME_LIMIT_HOURS


def _find_checkpoint_file(filename):
    if os.path.exists(filename) and os.path.getsize(filename) > 0:
        return filename
    input_base = "/kaggle/input"
    if os.path.exists(input_base):
        for root, _, files in os.walk(input_base):
            if filename in files:
                candidate = os.path.join(root, filename)
                if os.path.getsize(candidate) > 0:
                    return candidate
    return None


def _load_existing_progress(output_csv, fieldnames):
    if not RESUME:
        return set()
    src = _find_checkpoint_file(output_csv)
    if not src:
        return set()
    done_ids, rows = set(), []
    try:
        with open(src, "r", newline="", encoding="utf-8-sig") as f:
            reader = csv.DictReader(f)
            for r in reader:
                if "id" in r and r["id"]:
                    done_ids.add(r["id"])
                    rows.append(r)
        if src != output_csv and rows:
            with open(output_csv, "w", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writeheader()
                for r in rows:
                    writer.writerow({k: r.get(k, "") for k in fieldnames})
                f.flush()
            print(f"[RESUME] Loaded {len(done_ids):,} records from {src} -> {output_csv}")
        elif src == output_csv:
            print(f"[RESUME] Found existing {output_csv} with {len(done_ids):,} records")
    except Exception as e:
        print(f"[WARNING] Could not read checkpoint from {src}: {e}")
        return set()
    return done_ids


def run_piqa_evaluation(dataset, output_csv):
    """Run isQnA-aware Global PIQA evaluation on both script conditions."""
    done_ids = _load_existing_progress(output_csv, MCQ_FIELDNAMES)
    mode = "a" if (os.path.exists(output_csv) and os.path.getsize(output_csv) > 0) else "w"

    print(f"\n{'='*65}")
    print(f"Evaluating: Global PIQA (isQnA-aware) ({len(dataset):,} items, {len(done_ids):,} already done)")
    print(f"Output:     {output_csv}")
    print(f"{'='*65}")

    remaining = [it for it in dataset if it["id"] not in done_ids]
    if not remaining:
        print(f"All {len(dataset):,} items already completed.")
        return output_csv

    with open(output_csv, mode=mode, newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=MCQ_FIELDNAMES)
        if mode == "w":
            writer.writeheader()
            f.flush()

        t0 = time.time()
        n_processed = 0

        for i, item in enumerate(tqdm(remaining, desc="Global PIQA")):
            if _check_time_limit():
                elapsed_h = (time.time() - GLOBAL_START_TIME) / 3600.0
                print(f"\n[TIME LIMIT GUARD] Elapsed: {elapsed_h:.2f}h >= {TIME_LIMIT_HOURS}h. Saving and exiting.")
                break

            is_qna = item.get("isQnA", False)
            row = {"id": item["id"], "gold_label": item["label"], "is_qna": is_qna}

            for script in ["unicode", "romanized"]:
                s = item[script]
                prompt, digits = build_piqa_prompt(s["text"], s["options"], is_qna=is_qna)
                gen_text, n_in, n_out = generate_response(prompt)
                pred_digit = extract_mcq_choice(gen_text, digits)
                pred_label = digit_to_letter(pred_digit)
                correct = int(pred_label == item["label"])
                row[f"{script}_pred_digit"]      = pred_digit
                row[f"{script}_pred_label"]      = pred_label
                row[f"{script}_correct"]         = correct
                row[f"{script}_raw_output"]      = gen_text.strip()[:200]
                row[f"{script}_n_input_tokens"]  = n_in
                row[f"{script}_n_output_tokens"] = n_out

            writer.writerow(row)
            f.flush()
            n_processed += 1

        elapsed = time.time() - t0

    if n_processed:
        print(f"Done {n_processed:,} new items in {elapsed/60:.1f} min ({elapsed/n_processed:.2f} s/item)")
    return output_csv


print("Evaluation runner ready with continuous flushing, checkpointing, and time guard.")

Evaluation runner ready with continuous flushing, checkpointing, and time guard.


## 10. Run Evaluation

In [10]:
# ============================================================
# RUN GLOBAL PIQA EVALUATION
# ============================================================

piqa_csv = run_piqa_evaluation(
    piqa_data, f"{OUTPUT_PREFIX}_global_piqa.csv"
)

print("\n" + "="*65)
print("EVALUATION FINISHED")
print("="*65)


Evaluating: Global PIQA (isQnA-aware) (100 items, 0 already done)
Output:     lamini_gpt_1_5b_global_piqa.csv




Global PIQA:   0%|          | 0/100 [00:00<?, ?it/s]

Global PIQA:   1%|          | 1/100 [00:01<03:16,  1.98s/it]

Global PIQA:   2%|▏         | 2/100 [00:04<03:34,  2.18s/it]

Global PIQA:   3%|▎         | 3/100 [00:07<04:31,  2.80s/it]

Global PIQA:   4%|▍         | 4/100 [00:11<04:59,  3.12s/it]

Global PIQA:   5%|▌         | 5/100 [00:13<04:29,  2.84s/it]

Global PIQA:   6%|▌         | 6/100 [00:17<04:54,  3.14s/it]

Global PIQA:   7%|▋         | 7/100 [00:19<04:16,  2.76s/it]

Global PIQA:   8%|▊         | 8/100 [00:23<04:42,  3.07s/it]

Global PIQA:   9%|▉         | 9/100 [00:26<04:43,  3.12s/it]

Global PIQA:  10%|█         | 10/100 [00:28<04:20,  2.89s/it]

Global PIQA:  11%|█         | 11/100 [00:32<04:35,  3.09s/it]

Global PIQA:  12%|█▏        | 12/100 [00:34<04:02,  2.76s/it]

Global PIQA:  13%|█▎        | 13/100 [00:38<04:24,  3.03s/it]

Global PIQA:  14%|█▍        | 14/100 [00:40<04:01,  2.81s/it]

Global PIQA:  15%|█▌        | 15/100 [00:44<04:21,  3.08s/it]

Global P

Done 100 new items in 5.2 min (3.13 s/item)

EVALUATION FINISHED


## 11. Summary Statistics

In [11]:
import pandas as pd

def summarize_piqa(csv_path):
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        print("Global PIQA -- Not evaluated or file empty.")
        return None
    try:
        df = pd.read_csv(csv_path)
        n = len(df)
        if n == 0:
            print("Global PIQA n=0 (empty)")
            return None

        u_acc = df["unicode_correct"].mean() * 100
        r_acc = df["romanized_correct"].mean() * 100
        gap   = u_acc - r_acc
        u_inv = (df["unicode_pred_label"] == "INVALID").mean() * 100
        r_inv = (df["romanized_pred_label"] == "INVALID").mean() * 100

        print(f"\n{MODEL_LABEL} -- Global PIQA Evaluation (isQnA-aware T1_direct)")
        print("-" * 80)
        print(f"Overall  n={n:3d}  Unicode={u_acc:6.2f}%  Romanized={r_acc:6.2f}%  Gap={gap:+6.2f}pp"
              f"  Invalid(U/R)={u_inv:.1f}%/{r_inv:.1f}%")

        # Break down by isQnA
        for qna_val, lbl in [(True, "isQnA=True  (MCQ) "), (False, "isQnA=False (COMP)")]:
            sub = df[df["is_qna"] == qna_val]
            if len(sub) == 0:
                continue
            su = sub["unicode_correct"].mean() * 100
            sr = sub["romanized_correct"].mean() * 100
            sg = su - sr
            print(f"  {lbl}  n={len(sub):3d}  Unicode={su:6.2f}%  Romanized={sr:6.2f}%  Gap={sg:+6.2f}pp")

        result = {
            "Model":                  MODEL_LABEL,
            "N":                      n,
            "Unicode Acc (%)":         round(u_acc, 2),
            "Romanized Acc (%)":       round(r_acc, 2),
            "Gap (pp)":               round(gap, 2),
            "Unicode Invalid (%)":    round(u_inv, 2),
            "Romanized Invalid (%)":  round(r_inv, 2),
        }
        for qna_val, tag in [(True, "MCQ"), (False, "COMP")]:
            sub = df[df["is_qna"] == qna_val]
            if len(sub) > 0:
                result[f"Unicode {tag} Acc (%)"]   = round(sub["unicode_correct"].mean() * 100, 2)
                result[f"Romanized {tag} Acc (%)"] = round(sub["romanized_correct"].mean() * 100, 2)
        return result
    except Exception as e:
        print(f"[WARNING] Could not summarize {csv_path}: {e}")
        return None


result = summarize_piqa(piqa_csv)
if result:
    summary_csv = f"{OUTPUT_PREFIX}_global_piqa_summary.csv"
    pd.DataFrame([result]).to_csv(summary_csv, index=False)
    print(f"\nSummary saved to: {summary_csv}")
    display(pd.DataFrame([result])) if "display" in globals() else print(pd.DataFrame([result]))


LaMini-GPT-1.5B -- Global PIQA Evaluation (isQnA-aware T1_direct)
--------------------------------------------------------------------------------
Overall  n=100  Unicode=  0.00%  Romanized= 42.00%  Gap=-42.00pp  Invalid(U/R)=100.0%/19.0%
  isQnA=True  (MCQ)   n= 40  Unicode=  0.00%  Romanized= 37.50%  Gap=-37.50pp
  isQnA=False (COMP)  n= 60  Unicode=  0.00%  Romanized= 45.00%  Gap=-45.00pp

Summary saved to: lamini_gpt_1_5b_global_piqa_summary.csv
             Model    N  Unicode Acc (%)  Romanized Acc (%)  Gap (pp)  \
0  LaMini-GPT-1.5B  100              0.0               42.0     -42.0   

   Unicode Invalid (%)  Romanized Invalid (%)  Unicode MCQ Acc (%)  \
0                100.0                   19.0                  0.0   

   Romanized MCQ Acc (%)  Unicode COMP Acc (%)  Romanized COMP Acc (%)  
0                   37.5                   0.0                    45.0  


## 12. Output File Listing

In [12]:
print("Output files in working directory:")
for fpath in [piqa_csv]:
    if os.path.exists(fpath):
        size = os.path.getsize(fpath) / 1024
        print(f"  {fpath} ({size:.1f} KB)")
    else:
        print(f"  {fpath} (not created yet)")

summary_file = f"{OUTPUT_PREFIX}_global_piqa_summary.csv"
if os.path.exists(summary_file):
    size = os.path.getsize(summary_file) / 1024
    print(f"  {summary_file} ({size:.1f} KB)")

print(f"\n\u2705 {MODEL_LABEL} Global PIQA (isQnA-aware) evaluation complete!")

Output files in working directory:
  lamini_gpt_1_5b_global_piqa.csv (15.9 KB)
  lamini_gpt_1_5b_global_piqa_summary.csv (0.2 KB)

✅ LaMini-GPT-1.5B Global PIQA (isQnA-aware) evaluation complete!
